# Commerce embedding retrieval benchmark (Kaggle GPU)

Compare the proposal-listed BGE-M3, Qwen3 Embedding, and e5-large-v2 models over the manifest-allowlisted Commerce reference corpus and nine decision-domain retrieval cases. Attach a private asset bundle containing the public corpus JSONL, checksum metadata, current cases, and this evaluator; it contains no database credentials or private uploads. Enable **Internet** and a **GPU accelerator**.

Role filtering is applied before scoring. Each model runs in a fresh process with the same 512-token cap and batch size. This measures source-document ranking only; it does not measure answer quality or whether external practices are Commerce Demo policy.


In [ ]:
from pathlib import Path
import shutil

input_root = Path("/kaggle/input")
assets_candidates = sorted(
    project for project in input_root.rglob("project")
    if (project / "scripts" / "processing" / "evaluate_ocr.py").is_file()
    and (project / "scripts" / "retrieval" / "evaluate_embeddings.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected one attached Commerce embedding benchmark assets/project "
        "containing tools/retrieval/evaluate_embeddings.py; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent

In [ ]:
import hashlib, json

corpus = Path("data/evaluation/commerce_public_embedding_corpus.jsonl")
cases = Path("data/evaluation/commerce_retrieval_cases.json")
assert corpus.is_file() and cases.is_file()
metadata = json.loads(Path(str(corpus) + ".metadata.json").read_text())
assert hashlib.sha256(corpus.read_bytes()).hexdigest() == metadata["corpus_sha256"]
print(metadata)


In [ ]:
import subprocess, sys

# Keep the embedding framework version aligned with the repository lockfile.
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers==6.0.1"], check=True)
import torch
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU requested but CUDA is unavailable; restart with a Kaggle GPU accelerator.")

In [ ]:
import subprocess, sys

# Separate processes release model allocations between runs.
models = ["bge-m3", "qwen3-embedding", "e5-large-v2"]
results = Path("/kaggle/working/embedding-results")
results.mkdir(exist_ok=True)
for model in models:
    report = results / f"commerce-embedding-{model}-full-public-corpus.json"
    command = [sys.executable, "tools/retrieval/evaluate_embeddings.py", "--model", model, "--corpus", str(corpus), "--cases", str(cases), "--device", "cuda", "--batch-size", "4", "--max-chunks-per-document", "1000", "--top-k", "30", "--output", str(report)]
    subprocess.run(command, check=True)
    print(f"Saved {report}")


## Result files

Download the three JSON reports from `/kaggle/working/embedding-results/`. Each includes model revision, corpus/case hashes, GPU name, peak CPU/GPU memory, timings, role-filtered rankings, and per-case metrics. Keep the raw reports in `artifacts/evaluation/`; do not preserve corpus and case hashes because the labeled set may change.
